# Week 09 (training): Conditioning the diffusion model

In Week 08 you built and trained the unconditional diffusion model — a network that learns the *marginal* distribution of residuals across all training cycles, samples from it, and is ablated to confirm the timestep
embedding earns its keep. This week adds the missing ingredient:
**conditioning** on each window's smoothed sunspot area `area_smoothed`
and universal-path latitude `mu_universal`. With those two scalars
threaded into the network, the model goes from "produces plausible
residuals on average" to "produces residuals targeted at the specific
window the caller is asking about."

The architectural change is small. The Dataset returns a dictionary with keys `{"r_clean", "cond"}`
where cond is the 2-vector `(area_smoothed, mu_universal)`. The MLP gets a
slightly wider input — `r_t` and the timestep embedding *and* the
conditioning vector all concatenated at layer zero. The LightningModule's
training step unpacks the tuple and passes `cond` through. The sampler
accepts a per-sample conditioning vector and threads it through every
reverse step. None of the diffusion-specific machinery from Week 08 —
the schedule, the forward equation, the ε-prediction objective, the
timestep embedding itself — changes.

**By the end of this notebook you should have:**
- A trained conditional diffusion model that learns p(r | area, mu) — saved
  as `ckpt_conditional.ckpt`.
- A sanity-checked architecture confirming that t-sensitivity *and*
  cond-sensitivity both work: holding r_t fixed and varying t changes the
  output, *and* holding r_t and t fixed and varying cond also changes the
  output. The conditioning ablation built into Week 08 (the
  `use_timestep_embedding=False` flag) has a direct conditioning analogue
  here: if your model is somehow ignoring `cond`, the cond-sensitivity
  check is the only thing that will catch it before the evaluation
  notebook's distributional comparisons silently lie.

In [ ]:
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from einops import repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import wandb


In [ ]:
# ── locate Week 08/09 artifacts (split across two folders) ─────────────────
# The week-08 and week-09 deliverables are spread across weeks/week_08/
# and weeks/week_09/. We search both for each artifact independently so the
# notebook works regardless of where any given file sits.
def _find(filename, search_dirs):
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

_cwd = os.getcwd()
_search_dirs = []
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    for _sub in [("weeks", "week_09"), ("weeks", "week_08")]:
        _candidate = os.path.join(_base, *_sub)
        if os.path.isdir(_candidate) and _candidate not in _search_dirs:
            _search_dirs.append(_candidate)
    if ("week_08" in _base or "week_09" in _base) and os.path.isdir(_base) and _base not in _search_dirs:
        _search_dirs.append(_base)

_unconditioned_py  = _find("unconditioned_infrastructure.py", _search_dirs)
_conditioned_py    = _find("conditioned_infrastructure.py",   _search_dirs)
_parquet_path      = _find("diffusion_windows.parquet", _search_dirs)
_classical_py      = _find("butterflAI_model.py",     _search_dirs)
_classical_weights = _find("official_model.npz",      _search_dirs)

_missing = [n for n, p in [
    ("unconditioned_infrastructure.py", _unconditioned_py),
    ("conditioned_infrastructure.py",   _conditioned_py),
    ("diffusion_windows.parquet", _parquet_path),
    ("butterflAI_model.py",     _classical_py),
    ("official_model.npz",      _classical_weights),
] if p is None]
if _missing:
    raise FileNotFoundError(
        f"Cannot locate {_missing} under weeks/week_08 or weeks/week_09. "
        f"Searched: {_search_dirs}"
    )

_repo_root = os.path.abspath(os.path.join(os.path.dirname(_conditioned_py), "..", ".."))
for _p in [_repo_root,
           os.path.dirname(_unconditioned_py),
           os.path.dirname(_conditioned_py),
           os.path.dirname(_classical_py)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── import: Week 08 reused machinery + Week 09 conditional templates ───────
from unconditioned_infrastructure import (
    make_cosine_schedule,
    sinusoidal_embedding, TimestepEmbedding,
    SampleQualityCallback,
)
from conditioned_infrastructure import (
    ConditionalResidualDataset,
    ConditionalDiffusionMLP,
    ConditionalDiffusionLightning,
    sample_conditional,
)

# ── load Week 07 outputs ───────────────────────────────────────────────────
windows_df = pd.read_parquet(_parquet_path)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

# ── classical ButterflAI model (kept in scope for ad-hoc inspection) ───────
from butterflAI_model import ButterflAIModel
classical = ButterflAIModel(_classical_weights)

# ── cosine schedule (single source of truth from the Week 08 script) ───────
T = 200
alpha_np, sigma_np, _alpha_bar_np = make_cosine_schedule(T=T, s=0.008)

print(f"Loaded {len(windows_df)} windows from diffusion_windows.parquet")
print(f"  splits  : {windows_df['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles  : {sorted(windows_df['cycle'].unique())}")
print(f"  cond raw ranges (train split):")
_train = windows_df[windows_df['split'] == 'train']
for col in ['area_smoothed', 'mu_universal']:
    print(f"    {col:14s}: mean={_train[col].mean():.3f}, std={_train[col].std():.3f}, "
          f"min={_train[col].min():.3f}, max={_train[col].max():.3f}")
print(f"  schedule: T={T}, arrays length {len(alpha_np)}")
print(classical)

---
## Task 45 — `ConditionalResidualDataset` (template in `conditioned_infrastructure.py`)

The new Dataset returns a **dict** instead of a tuple:

```
{"r_clean": float32 tensor (15,),
 "cond":    float32 tensor (2,)}
```

Dict-returning Datasets are the convention worth adopting from Week 09
onward. Dicts are self-documenting at access time (`batch["cond"]` needs
no memory of position), extensible (adding a `cycle_phase` field later
does not break unpacking sites in the training step, the sampler, or the
evaluation notebook — only the consumers that actually use the new field
need updating), and PyTorch's default DataLoader collation handles
dict-of-tensors natively. The tuple-returning version of this dataset
would work too, but every new conditioning variable would require
chasing down every `r, cond = batch` line in the codebase. The dict
pattern scales; the tuple pattern does not.

Two pieces of standardization are still happening, exactly as discussed
in the script's docstring:

**Residuals** are standardized per-bin to unit variance, exactly as in
Week 08. Each split computes its own `bin_means` / `bin_stds`. The
LightningModule persists the train-split bin statistics as buffers so the
sampler can de-standardize at inference time.

**Conditioning** is standardized using *train-split-only* statistics —
this is the part that requires care. The network learns to expect cond
inputs in the train-set's normalized scale. If the validation dataset
standardized using its own statistics, the network would see
distributionally different conditioning at evaluation time than it saw at
training time, which silently corrupts every cond-sensitivity check
downstream.

The template handles this for you: when you construct any
`ConditionalResidualDataset` without passing `cond_means` / `cond_stds`,
the constructor computes them from the *train rows of the supplied
DataFrame* regardless of which split the Dataset itself represents. So
the `val` dataset gets the train-set normalization automatically.

**Open `conditioned_infrastructure.py` and implement:**
- `ConditionalResidualDataset.__init__` (residual standardization +
  conditioning standardization with the train-split-only constraint).
- `ConditionalResidualDataset.__len__`.
- `ConditionalResidualDataset.__getitem__` (returns the dict described
  above with the right keys, dtypes, and shapes).


---
## Task 46 — Visualize the conditional dataset

A dataset class is one of the easier objects to silently misimplement.
For the conditional version there is *one more thing* to check beyond
the Week 08 sanity tests: that conditioning standardization actually uses
train-split statistics for the validation set. If your val dataset has
cond mean far from zero or cond std far from one — but the *train*
dataset does have cond mean zero and cond std one — you accidentally
standardized each split with its own stats, which is the failure mode
this Dataset spec was designed to prevent.

**Tasks:**
- Instantiate `train_dataset` and `val_dataset`. Print their lengths.
  The counts should match the splits printed by the setup cell.
- Pull `train_dataset[0]` and verify it is a **dict** with the right keys
  (`"r_clean"`, `"cond"`), each value a float32 tensor of the expected
  shape, finite. If you accidentally returned a tuple from `__getitem__`,
  this is where you find out.
- Print the `cond_means` and `cond_stds` of both datasets. They must be
  **identical** across train and val. If they differ, the
  train-split-only rule was violated.
- For the train dataset, verify that the cond mean is ≈ 0 and the cond
  std is ≈ 1 (both within ~1e-5). For val, cond mean and std will be
  near these but not exactly — you are applying train statistics to a
  different distribution, so deviations from 0/1 are expected and
  meaningful.
- Plot the raw `(area_smoothed, mu_universal)` joint distribution coloured
  by split, then the normalized joint distribution. The normalized
  version should be centred on (0, 0) with the train cloud reaching
  roughly ±2 in each coordinate.


In [ ]:
# Task 46 — instantiate, sanity-check, visualize the conditional datasets

# 1. Instantiate both datasets (train computes stats; val inherits them).
raise NotImplementedError("Task 46: instantiate train/val ConditionalResidualDatasets, then implement the sanity checks and visualizations specified above.")

# Reference workflow once Task 46 is implemented:
#
#   train_dataset = ConditionalResidualDataset(windows_df, "train")
#   val_dataset   = ConditionalResidualDataset(windows_df, "val")
#
#   # Verify both datasets share the same cond standardization.
#   assert torch.allclose(train_dataset.cond_means, val_dataset.cond_means)
#   assert torch.allclose(train_dataset.cond_stds,  val_dataset.cond_stds)
#
#   # Then: print stats, plot raw and normalized cond distributions per split.


---
## Task 47 — Build the DataLoaders

Same DataLoader pattern as Week 08. PyTorch's default collation handles
the Dataset's dict return automatically: each key in the per-item dict
becomes a key in the per-batch dict, with the leading batch dimension
prepended to every tensor value. So `next(iter(train_loader))` is a
dict:

```
{"r_clean": (B, 15) float32, "cond": (B, 2) float32}
```

Verify this explicitly — the LightningModule's `_shared_step` will read
from these keys, and a tuple-returning Dataset (from accidentally writing
`return r, cond` instead of `return {"r_clean": r, "cond": cond}` in
`__getitem__`) will give batches that look superficially similar but
fail in the training step several layers deeper.


In [ ]:
# Task 47 — DataLoaders + batch sanity check.

BATCH_SIZE = 64

raise NotImplementedError("Task 47: build train_loader and val_loader with the same conventions as Week 08, then verify next(iter(loader)) returns a dict with keys 'r_clean' and 'cond' and the expected shapes/dtypes.")

# Reference workflow:
#
#   train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE,
#                                              shuffle=True,  num_workers=0)
#   val_loader   = torch.utils.data.DataLoader(val_dataset,   batch_size=BATCH_SIZE,
#                                              shuffle=False, num_workers=0)
#
#   for name, loader in [("train", train_loader), ("val", val_loader)]:
#       batch = next(iter(loader))
#       assert isinstance(batch, dict), f"expected dict, got {type(batch).__name__}"
#       assert set(batch.keys()) == {"r_clean", "cond"}, f"wrong keys: {set(batch.keys())}"
#       assert batch["r_clean"].shape[1] == 15 and batch["cond"].shape[1] == 2
#       assert batch["r_clean"].dtype == torch.float32
#       assert batch["cond"].dtype    == torch.float32
#       print(f"{name:5s}: {len(loader):3d} batches/epoch, "
#             f"r_clean shape={tuple(batch['r_clean'].shape)}, "
#             f"cond shape={tuple(batch['cond'].shape)}")


---
## Task 48 — `ConditionalDiffusionMLP` (template in `conditioned_infrastructure.py`)

The forward signature changes from `(r_t, t) → eps_hat` to
`(r_t, t, cond) → eps_hat`. The conditioning vector is concatenated
alongside r_t and the timestep embedding at the input layer:

```
x = torch.cat([r_t, t_emb, cond], dim=-1)   # shape (B, 15 + 128 + 2)
```

No new architectural concepts beyond that. The `TimestepEmbedding` is
reused unchanged from Week 08, imported from `unconditioned_infrastructure.py`.

**Open `conditioned_infrastructure.py` and implement** the `__init__` and
`forward` of `ConditionalDiffusionMLP`. Default constructor arguments
are pinned to match what the rest of this notebook expects.

---
## Task 49 — Sanity tests: t-sensitivity *and* cond-sensitivity

Week 08 introduced the t-sensitivity check: hold r_t fixed, vary t,
verify the output measurably changes. That check defends against a
silently-broken timestep embedding.

The conditional model needs the analogous check for the conditioning:
**cond-sensitivity** — hold r_t and t fixed, vary `cond` across the
range of (area, mu) values the model will actually see, verify the output
measurably changes. If it does not, your model is silently unconditional
no matter what the class name says, and every distributional comparison
in the evaluation notebook will give answers that look reasonable but
are not measuring what you think they are.

Run **both** checks on a freshly-initialized model before training. The
checks are inexpensive; the failure mode they catch is exactly the kind
of silent corruption that wastes days of training time.

For the cond-sensitivity check, use a *realistic* range of conditioning
values — sample five (area, mu) points spanning roughly ±2 standard
deviations of the train distribution (the normalized range the model
will actually encounter), not arbitrary synthetic values. This makes the
check both more meaningful and a closer analogue of the t-sensitivity
check, which uses real timestep values.

In [ ]:
# Task 49 — t-sensitivity + cond-sensitivity on a fresh model.

raise NotImplementedError("Task 49: run the t-sensitivity check (as in Week 08) and the new cond-sensitivity check on a freshly-initialized ConditionalDiffusionMLP. Both must show measurably non-zero pairwise differences in the network's outputs; if either is exactly zero, the corresponding input is being silently dropped.")

# Reference workflow:
#
#   torch.manual_seed(1)
#   model_test = ConditionalDiffusionMLP()
#
#   r_t_fixed = torch.randn(15)
#   t_values  = torch.tensor([0, T//4, T//2, 3*T//4, T-1], dtype=torch.long)
#   cond_zero = torch.zeros(2)                                     # mid-range cond
#
#   # 1. t-sensitivity: hold r_t and cond fixed, vary t.
#   r_t_batch  = repeat(r_t_fixed, "d -> n d", n=5)
#   cond_batch = repeat(cond_zero, "d -> n d", n=5)
#   out_t = model_test(r_t_batch, t_values, cond_batch)
#   #    Verify out_t[i] differs from out_t[j] for i != j.
#
#   # 2. cond-sensitivity: hold r_t and t fixed, vary cond.
#   #    Sample 5 cond vectors spanning ±2 in normalized (area, mu) space.
#   cond_values = torch.tensor([[-2., -2.], [-1., 1.], [0., 0.], [1., -1.], [2., 2.]])
#   t_fixed = torch.full((5,), T // 2, dtype=torch.long)
#   r_t_batch = repeat(r_t_fixed, "d -> n d", n=5)
#   out_c = model_test(r_t_batch, t_fixed, cond_values)
#   #    Verify out_c[i] differs from out_c[j] for i != j.
#
#   # Plot two 5x5 pairwise-L2-distance heatmaps side by side: t-sensitivity
#   # on the left, cond-sensitivity on the right. Both should be visibly
#   # non-zero off-diagonal. If either is exactly zero, the corresponding
#   # input is being silently dropped.

---
## Task 50 — `ConditionalDiffusionLightning` (template in `conditioned_infrastructure.py`)

The only structural changes from Week 08's `DiffusionLightning`:

1. `_shared_step` reads `batch["r_clean"]` and `batch["cond"]` from the
   dict batch (rather than unpacking a tuple) and passes `cond` through
   to `self.model(r_t, t, cond)`.
2. The constructor accepts `cond_means` and `cond_stds` and registers them
   as buffers alongside the existing `bin_means` / `bin_stds`. This keeps
   *all* relevant standardization statistics on the checkpoint, so the
   evaluation notebook can load and re-normalize conditioning without
   needing the training-time dataset.
3. `save_hyperparameters(ignore=[...])` now ignores the four statistics
   tensors alongside `model`, `alpha`, `sigma`.

The `configure_optimizers` method is *identical* to Week 08's. Copy it
from `unconditioned_infrastructure.py`.

**Open `conditioned_infrastructure.py` and implement** `__init__`,
`_shared_step`, and `configure_optimizers`. The `training_step` and
`validation_step` are filled in for you — they call `_shared_step` and
log loss exactly as in Week 08.

---
## Task 51 — Train the conditional model


In [ ]:
# Task 51 — Build the standardized-residual + de-standardized arrays
#           the SampleQualityCallback needs, then train.

raise NotImplementedError("Task 51: assemble (a) `all_train_phys`, the de-standardized residual array used by SampleQualityCallback for distributional comparisons during training, and (b) the de-standardized val analogue. Then instantiate the model, the LightningModule (passing all four standardization statistics), the WandbLogger (with CSVLogger fallback), the SampleQualityCallback, the Trainer, and fit. Save a single checkpoint as './ckpt_conditional.ckpt'.")

# Reference workflow:
#
#   from infrastructure.utils.reproducibility import set_all_seeds
#   set_all_seeds(42)
#
#   # De-standardize for the SampleQualityCallback (same convention as Week 08).
#   # Each Dataset item is now a dict, so read by key.
#   bin_means_np = train_dataset.bin_means.numpy()
#   bin_stds_np  = train_dataset.bin_stds.numpy()
#   all_train      = torch.stack([train_dataset[i]["r_clean"] for i in range(len(train_dataset))]).numpy()
#   all_val        = torch.stack([val_dataset[i]["r_clean"]   for i in range(len(val_dataset))]).numpy()
#   all_train_phys = all_train * bin_stds_np + bin_means_np
#   all_val_phys   = all_val   * bin_stds_np + bin_means_np
#
#   MAX_EPOCHS = 10000
#
#   model_cond = ConditionalDiffusionMLP()
#   lightning_cond = ConditionalDiffusionLightning(
#       model=model_cond, alpha=alpha_np, sigma=sigma_np, T=T, lr=1e-3,
#       scheduler="cosine", weight_decay=1e-4,
#       bin_means=train_dataset.bin_means, bin_stds=train_dataset.bin_stds,
#       cond_means=train_dataset.cond_means, cond_stds=train_dataset.cond_stds,
#   )
#
#   try:
#       logger_cond = WandbLogger(
#           project="butterflai-wk09",
#           name=f"conditional_b{BATCH_SIZE}_e{MAX_EPOCHS}_cos",
#           save_dir="./wandb_logs",
#       )
#   except Exception as _e:
#       print(f"WandB unavailable ({_e}); using CSVLogger.")
#       logger_cond = CSVLogger("./csv_logs", name="conditional")
#
#   sample_quality_cb = SampleQualityCallback(
#       train_samples=all_train_phys, val_samples=all_val_phys,
#       every_n_epochs=100, n_compare=500,
#       bin_centers=BIN_CENTERS, bin_width=BIN_WIDTH,
#   )
#
#   trainer_cond = pl.Trainer(
#       max_epochs=MAX_EPOCHS, logger=logger_cond,
#       accelerator="auto", devices="auto",
#       log_every_n_steps=10, enable_progress_bar=False,
#       callbacks=[sample_quality_cb],
#   )
#
#   trainer_cond.fit(lightning_cond, train_loader, val_loader)
#   trainer_cond.save_checkpoint("./ckpt_conditional.ckpt")
#   try: wandb.finish()
#   except Exception: pass
#
# IMPORTANT: SampleQualityCallback was written for unconditional sampling.
# Inside its on_train_epoch_end and on_train_end, it calls
# `sample(pl_module, ...)`. That function does not exist on the conditional
# module — sampling is `sample_conditional`, which requires a `cond`
# argument. If you keep the Week 08 callback unchanged, the periodic
# diagnostics will fail with an AttributeError or a missing-argument error
# the first time they fire. Two ways out:
#
#   1. Subclass SampleQualityCallback and override `on_train_epoch_end` and
#      `on_train_end` to call `sample_conditional` with a fixed batch of
#      reference conditioning vectors (e.g., drawn from the train set).
#   2. Drop the callback entirely from this run and rely on the loss curves
#      + the evaluation notebook for distributional checks.
#
# Either is fine for Week 09; option 1 is the more pedagogically useful
# extension if you have the bandwidth.


---
## Handoff to `09d_conditioned_evaluate.ipynb`

You should now have one checkpoint file in this directory:
- `ckpt_conditional.ckpt` (conditional diffusion model)

Together with the Week 08 unconditional checkpoint `ckpt_full.ckpt`, this
gives the evaluation notebook two trained models to compare. The
evaluation notebook will:

- Reload both checkpoints and re-verify their t-sensitivity (and, for the
  conditional model, cond-sensitivity) on the *loaded* state — defends
  against silent state-dict corruption during loading.
- Sample residuals from the conditional model targeted at specific
  validation-window conditioning, visualize them.
- Compare the conditional samples' distribution to held-out validation
  residuals, against the unconditional baseline.

The headline value-add metric — `compute_global_nll` on held-out cycles,
classical alone vs. classical + conditional residuals — is its own
notebook (`09e_diffusion_NLL_evaluation.ipynb`). That's the question
the program has been pointing at since Week 03, and it gets its own
clean space to be answered.

**If you change anything in `conditioned_infrastructure.py` after this notebook
has been run**, the autoreload magic in the evaluation notebook will pick
it up — but checkpoints saved with the old code will not be compatible
with new class signatures. If you change a class signature, retrain.
